In [2]:
import numpy as np
import pandas as pd

train_df = pd.read_csv("..\DATASET\stage2_train_resampled.csv")
val_df   = pd.read_csv("..\DATASET\stage2_val_clean.csv")
test_df  = pd.read_csv("..\DATASET\stage2_test_clean.csv")

X_train = train_df.drop(columns=['Label'])
y_train = train_df['Label']

X_val = val_df.drop(columns=['Label'])
y_val = val_df['Label']

X_test = test_df.drop(columns=['Label'])
y_test = test_df['Label']

<>:4: SyntaxWarning: invalid escape sequence '\D'
<>:5: SyntaxWarning: invalid escape sequence '\D'
<>:6: SyntaxWarning: invalid escape sequence '\D'
<>:4: SyntaxWarning: invalid escape sequence '\D'
<>:5: SyntaxWarning: invalid escape sequence '\D'
<>:6: SyntaxWarning: invalid escape sequence '\D'
C:\Users\sudee\AppData\Local\Temp\ipykernel_14144\288841998.py:4: SyntaxWarning: invalid escape sequence '\D'
  train_df = pd.read_csv("..\DATASET\stage2_train_resampled.csv")
C:\Users\sudee\AppData\Local\Temp\ipykernel_14144\288841998.py:5: SyntaxWarning: invalid escape sequence '\D'
  val_df   = pd.read_csv("..\DATASET\stage2_val_clean.csv")
C:\Users\sudee\AppData\Local\Temp\ipykernel_14144\288841998.py:6: SyntaxWarning: invalid escape sequence '\D'
  test_df  = pd.read_csv("..\DATASET\stage2_test_clean.csv")


In [3]:
from sklearn.preprocessing import StandardScaler

In [4]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

In [5]:
from sklearn.decomposition import PCA

pca = PCA(n_components=20)

X_train_pca = pca.fit_transform(X_train_scaled)
X_val_pca   = pca.transform(X_val_scaled)
X_test_pca  = pca.transform(X_test_scaled)

In [5]:
from sklearn.cluster import KMeans

n_clusters = len(y_train.unique())  # e.g., 4 attack types

kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)

kmeans.fit(X_train_pca)

KMeans(n_clusters=4, n_init=10, random_state=42)

In [6]:
import numpy as np
from scipy.stats import mode

train_clusters = kmeans.predict(X_train_pca)

cluster_to_label = {}

for i in range(n_clusters):
    mask = (train_clusters == i)
    
    if np.sum(mask) == 0:
        continue
        
    cluster_to_label[i] = mode(y_train[mask], keepdims=True)[0][0]

print("Cluster → Label mapping:", cluster_to_label)

Cluster → Label mapping: {0: 2, 1: 1, 2: 3, 3: 4}


In [7]:
def predict_stage2(X_scaled):
    clusters = kmeans.predict(X_scaled)
    return np.array([cluster_to_label.get(c, -1) for c in clusters])

In [8]:
from sklearn.metrics import classification_report, f1_score

y_val_pred = predict_stage2(X_val_pca)

print("Validation Report:")
print(classification_report(y_val, y_val_pred))

print("Val F1:", f1_score(y_val, y_val_pred, average='macro'))

Validation Report:
              precision    recall  f1-score   support

           1       1.00      0.75      0.86     28951
           2       0.78      1.00      0.87      8173
           3       0.10      0.75      0.18       824
           4       1.00      0.00      0.01       373

    accuracy                           0.80     38321
   macro avg       0.72      0.63      0.48     38321
weighted avg       0.93      0.80      0.84     38321

Val F1: 0.47881406781513813


In [9]:
y_test_pred = predict_stage2(X_test_pca)

print("Test Report:")
print(classification_report(y_test, y_test_pred))

print("Test F1:", f1_score(y_test, y_test_pred, average='macro'))

Test Report:
              precision    recall  f1-score   support

           1       1.00      0.75      0.86     32177
           2       0.78      1.00      0.88      9082
           3       0.10      0.77      0.18       915
           4       1.00      0.00      0.00       414

    accuracy                           0.80     42588
   macro avg       0.72      0.63      0.48     42588
weighted avg       0.93      0.80      0.84     42588

Test F1: 0.4809732338399456


In [10]:


print("Test distribution:\n", pd.Series(y_test_pred).value_counts())
print("\nTest actual:\n", pd.Series(y_test).value_counts())

from sklearn.metrics import f1_score
print("\nTest F1:", f1_score(y_test, y_test_pred, average='macro'))

Test distribution:
 1    24236
2    11539
3     6812
4        1
Name: count, dtype: int64

Test actual:
 Label
1    32177
2     9082
3      915
4      414
Name: count, dtype: int64

Test F1: 0.4809732338399456


In [ ]:
from sklearn.neighbors import NearestNeighbors

from sklearn.neighbors import NearestNeighbors
import numpy as np
import matplotlib.pyplot as plt

neighbors = NearestNeighbors(n_neighbors=10)
neighbors_fit = neighbors.fit(X_train_pca)

distances, _ = neighbors_fit.kneighbors(X_train_pca)

distances = np.sort(distances[:, -1])

plt.plot(distances)
plt.title("k-distance graph")
plt.show()

In [7]:
def predict_dbscan(X_pca):
    distances, indices = nn.kneighbors(X_pca)
    
    preds = []
    
    for idx in indices.flatten():
        cluster = train_clusters[idx]
        
        if cluster == -1:
            preds.append(-1)
        else:
            preds.append(cluster_to_label.get(cluster, -1))
    
    return np.array(preds)

In [9]:
from sklearn.metrics import classification_report, f1_score

y_val_pred = predict_dbscan(X_val_pca)

print("Validation Report:")
print(classification_report(y_val, y_val_pred))

print("Val F1:", f1_score(y_val, y_val_pred, average='macro'))

NameError: name 'train_clusters' is not defined